# Ollama + Gradio chat

Requirements: [Ollama](https://ollama.com) installed and running (`ollama serve`, or the desktop app).

In [1]:
%pip install -q ollama gradio

/Users/laguardia/Documents/repos/agent-mock/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import ollama

MODEL = "llama3.2:1b"  # very light (~1.3GB). Alternatives: "qwen2.5:0.5b" (~400MB), "gemma3:1b"

# Download the model if you don't have it yet (no-op if already present)
ollama.pull(MODEL)
print("Model ready:", MODEL)

Model ready: llama3.2:1b


In [3]:
import gradio as gr

def chat(message, history):
    # history is a list of {"role": ..., "content": ...} dicts
    def to_text(content):
        # Gradio may give a str or a list of blocks like [{"type": "text", "text": "..."}]
        if isinstance(content, str):
            return content
        if isinstance(content, dict):
            content = [content]
        return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)

    messages = [{"role": m["role"], "content": to_text(m["content"])} for m in history]
    messages.append({"role": "user", "content": message})

    reply = ""
    for chunk in ollama.chat(model=MODEL, messages=messages, stream=True):
        reply += chunk["message"]["content"]
        yield reply

demo = gr.ChatInterface(
    chat,
    title=f"Chat with {MODEL}",
    examples=["Explain what an API is in two sentences.", "Write a haiku about Python."],
)

demo.launch()  # shows inline in the notebook; open the printed URL for a full tab

/Users/laguardia/Documents/repos/agent-mock/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
